# Retraining the two-stage Lasso + random forest model on the train + validation rows

The evaluation plan, for the model `two_stage_lasso_rf` (a Lasso for the monthly level, a random forest for the flat):

1. The **train-only model** (`two_stage_models/two_stage_lr_forest.ipynb`, fitted on the train months 2021-01..2024-03) is scored on the held-out validation rows.
2. **This notebook** refits the same model on the train + validation rows (2021-01..2025-03, 149,999 rows).
3. The test predictions of the **retrained** model are the ones sent to Kaggle.

Nothing here overwrites the train-only model or its predictions: everything gets the prefix `retrained_`, as for the neural networks (`retrain_neural_networks.ipynb`).

* **Settings.** Nothing is tuned again: the Lasso penalty `alpha` and the forest settings are read from the saved train-only model (`two_stage_models/results/models/two_stage_lasso_rf.joblib`). They were chosen on the validation rows, which are now part of the fit.
* **Stage 1 (level by month).** The Lasso on the 177 temporal features is refitted on the 51 months 2021-01..2025-03, each month weighted by its number of rows, with the rent standardised by the mean and standard deviation of all 149,999 rents (exactly as the train-only fit does for its rows).
  The selection of temporal features can change with the extra 12 months; the notebook lists how many are kept and how many are shared with the train-only model.
* **Stage 2 (the flat).** The forest is refitted on all 149,999 rows: target = rent / (in-sample level of its month from the retrained Lasso), weighted by `level^2`. The scalers of the data pipeline stay as they were fitted (on the train months), so the inputs are on the same scale as for the train-only model.
* **Predictions.** Same files and row sets as the train-only model, so the evaluation code reads them unchanged: `train/retrained_two_stage_lasso_rf.csv` holds the 114,841 train rows, `val/` the 35,158 validation rows, `test/` the 50,000 test rows, `levels/` the fixed-flat level of each month.
  **The retrained model was fitted on both the train and the validation rows, so both of those files are in-sample.** The validation scores of the train-only model are the only held-out accuracy.
* **Outputs.** The model and the lists of kept features in `results/models/` and `results/tables/retrained_two_stage_lasso_rf/`, the predictions in `03_evaluation/predictions/`.

Run it from this folder, after `two_stage_lr_forest.ipynb` has produced the train-only model.

In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso

sys.path.insert(0, str(Path.cwd().parent))          # 02_modelling: nn_utils.py
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
NAME = 'two_stage_lasso_rf'
RETRAINED = 'retrained_' + NAME
TRAIN_ONLY_MODEL = Path.cwd().parent / 'two_stage_models' / 'results' / 'models' / f'{NAME}.joblib'
nu.use_results_dir(Path.cwd() / 'results')
nu.ensure_output_dirs(RETRAINED)

## 1. Data and the settings of the train-only model

The same rows as the train-only notebook. All 149,999 rows with a rent (`train` and `val`) are fitted; the `sample_type` flags are only used to split the predictions into the `train` and `val` files.
The Lasso penalty and the forest settings are read from the saved train-only model, together with the feature lists, which must be the current ones.

In [2]:
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)
fixed = np.isin(month, nu.Q1_2025)                                                   # the fixed flats: the rows of 2025-01..03 (their features only, never their rents)
temporal = nu.load_temporal()                                                        # 67 months x 177 scaled features

old = joblib.load(TRAIN_ONLY_MODEL)
assert old['temporal_features'] == list(temporal.columns) and old['static_features'] == STATIC, 'the train-only model was fitted on other features: rerun two_stage_lr_forest.ipynb'
ALPHA, RF_PARAMS = old['alpha'], old['rf_params']
print(f'{len(STATIC)} static + {temporal.shape[1]} temporal features; retraining on {len(y):,} rows (months 0..{month.max()}): {fit.sum():,} train rows + {val.sum():,} validation rows; test rows {len(test):,}')
print(f'settings of the train-only model: alpha = {ALPHA:.6f}, forest {RF_PARAMS}')

151 static + 177 temporal features; retraining on 149,999 rows (months 0..50): 114,841 train rows + 35,158 validation rows; test rows 50,000
settings of the train-only model: alpha = 0.000464, forest {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6, 'max_features': 0.33, 'min_samples_leaf': 10}


## 2. Stage 1: the Lasso on the months 0..50

The same function as in the train-only notebook, applied to all the rows instead of the train rows.

In [3]:
def fit_lasso(alpha, rows):
    """Lasso of the standardised monthly mean rent on the temporal features, each month weighted by its number of rows (the same as the Lasso on the rows)."""
    counts = pd.Series(month[rows]).value_counts().sort_index()
    mean_rent = pd.Series(y[rows]).groupby(month[rows]).mean()
    y_mean, y_std = y[rows].mean(), y[rows].std()
    lasso = Lasso(alpha=alpha, max_iter=200_000).fit(temporal.loc[mean_rent.index].to_numpy(), (mean_rent.to_numpy() - y_mean) / y_std, sample_weight=counts.to_numpy(dtype=float))
    return {'lasso': lasso, 'y_mean': y_mean, 'y_std': y_std, 'alpha': alpha}


model = fit_lasso(ALPHA, np.ones(len(y), dtype=bool))
level = pd.Series(model['lasso'].predict(temporal.to_numpy()) * model['y_std'] + model['y_mean'], index=temporal.index)       # stage-1 level of the 67 months
level_w = level.loc[month].to_numpy()

coefficients = pd.Series(model['lasso'].coef_, index=temporal.columns)
selected = coefficients[coefficients != 0].sort_values(key=np.abs, ascending=False).rename('coefficient (standardised rent per scaled feature)')
kept_old = set(temporal.columns[old['lasso'].coef_ != 0])
print(f'temporal features kept: {len(selected)} (train-only model: {len(kept_old)}); shared with the train-only model: {len(kept_old & set(selected.index))}')
print(selected.head(10).round(4).to_string())
print(level.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level').T.to_string())

temporal features kept: 50 (train-only model: 40); shared with the train-only model: 27
MACRO_SORA_3M_PCT                               0.2364
stk_sheng_siong_group_log_price_mean            0.0532
stk_singapore_airlines_log_price_mean           0.0372
MACRO_UNEMPLOYMENT_SA_PCT                      -0.0348
stk_singapore_post_log_price_mean              -0.0341
stk_sea_limited_log_price_mean                 -0.0315
coe_c_quota_premium                             0.0267
stk_sats_log_price_mean                        -0.0243
stk_wilmar_international_log_price_mean        -0.0222
stk_mapletree_logistics_trust_log_price_mean   -0.0217
       2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level   3114.0   3123.0   3146.0   3129.0   3125.0   3137.0   3162.0   3219.0   3278.0   3303.0   3328.0   3366.0   3333.0   3354.0   3463.0   3456.0

## 3. Stage 2: the forest on all the rows

The forest with the train-only settings, fitted on the rows of all 51 months: target = rent / (in-sample level of its month), weighted by `level^2`.
Prediction = `level of the month x ratio of the flat`; the level of the same flats in each month is level(month) x the mean ratio of the fixed flats.

In [4]:
forest = RandomForestRegressor(n_jobs=-1, random_state=0, **RF_PARAMS).fit(X, y / level_w, sample_weight=level_w ** 2)
ratio_all, ratio_test = forest.predict(X), forest.predict(X_test)
pred_all = level_w * ratio_all
pred = {'train': pred_all[fit], 'val': pred_all[val], 'test': level.loc[test['month']].to_numpy() * ratio_test}
level_fixed = level * forest.predict(X[fixed]).mean()
print(f'in-sample ratio mean {ratio_all.mean():.3f}; test ratio mean {ratio_test.mean():.3f}')
print(f'in-sample RMSE: train rows ${nu.rmse(y[fit], pred["train"]):,.1f}, validation rows ${nu.rmse(y[val], pred["val"]):,.1f} (the validation rows were fitted: not a held-out score)')
print(level_fixed.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

in-sample ratio mean 1.000; test ratio mean 1.001
in-sample RMSE: train rows $451.1, validation rows $440.0 (the validation rows were fitted: not a held-out score)
                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3113.0   3123.0   3145.0   3128.0   3124.0   3137.0   3162.0   3219.0   3277.0   3303.0   3327.0   3365.0   3333.0   3354.0   3462.0   3456.0   3473.0   3514.0   3529.0   3624.0   3651.0   3667.0   3666.0


## 4. Save the model, the kept features and the predictions

In [5]:
paths = [nu.MODEL_DIR / f'{RETRAINED}.joblib']
joblib.dump({'lasso': model['lasso'], 'target_mean': model['y_mean'], 'target_std': model['y_std'], 'alpha': ALPHA, 'temporal_features': list(temporal.columns), 'forest': forest, 'static_features': STATIC,
             'rf_params': RF_PARAMS, 'target': 'rent / stage-1 level of the month, weighted by level^2',
             'note': 'refitted on the train + validation rows (2021-01..2025-03) with the alpha and the forest settings of the train-only model'}, paths[0], compress=3)
paths += [nu.save_table(selected, RETRAINED, 'selected_temporal_features.csv', index_label='feature')]
summary = pd.DataFrame([{'model': RETRAINED, 'rows_fitted': len(y), 'months_fitted': int(len(np.unique(month))), 'alpha': ALPHA, 'temporal_features_kept': len(selected), 'shared_with_train_only': len(kept_old & set(selected.index)),
                         'in_sample_rmse_train_rows': nu.rmse(y[fit], pred['train']), 'in_sample_rmse_validation_rows': nu.rmse(y[val], pred['val'])}])
paths += [nu.save_table(summary, RETRAINED, 'summary.csv', index=False)]
paths += [nu.save_predictions(RETRAINED, 'train', train[fit], pred['train']),          # the 114,841 train rows (in-sample)
          nu.save_predictions(RETRAINED, 'val', train[val], pred['val']),              # the 35,158 validation rows (in-sample too: they were fitted)
          nu.save_predictions(RETRAINED, 'test', test, pred['test']),                  # the 50,000 test rows (a forecast)
          nu.save_levels(RETRAINED, level_fixed)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_retraining/results/models/retrained_two_stage_lasso_rf.joblib
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_retraining/results/tables/retrained_two_stage_lasso_rf/selected_temporal_features.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_retraining/results/tables/retrained_two_stage_lasso_rf/summary.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/train/retrained_two_stage_lasso_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/retrained_two_stage_lasso_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/test/retrained_two_stage_lasso_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/levels/retrained_two_

## 5. Check against the train-only predictions

The retrained files must cover the same rows in the same order as the train-only ones (the evaluation matches models by `ROW_ID`). The mean test prediction of the retrained model should sit near the train-only one.

In [6]:
rows = []
for split in nu.SPLITS:
    a = pd.read_csv(nu.PRED_DIR / split / f'{NAME}.csv', usecols=['ROW_ID', 'prediction'])
    b = pd.read_csv(nu.PRED_DIR / split / f'{RETRAINED}.csv', usecols=['ROW_ID', 'prediction'])
    assert len(a) == len(b) and (a['ROW_ID'].to_numpy() == b['ROW_ID'].to_numpy()).all(), split
    rows.append({'split': split, 'rows': len(b), 'mean_train_only': a['prediction'].mean(), 'mean_retrained': b['prediction'].mean(), 'rmse_between_the_two': nu.rmse(a['prediction'], b['prediction'])})
print(pd.DataFrame(rows).round(1).to_string(index=False))

split   rows  mean_train_only  mean_retrained  rmse_between_the_two
train 114841           2593.6          2593.2                  48.0
  val  35158           3144.5          3126.7                  75.8
 test  50000           3256.0          3449.6                 219.5
